# Pig-English zh-TW TTS 生成器

用 Microsoft Edge TTS（Azure neural voice）一次性生成 pig-english 需要的中文 mp3。

**輸入**：pig-english 的 `vocabulary.ts`（已 export 為共用 repo 的 `data/vocab-en-preschool/words.json`）。
**輸出**：3 voice × (152 vocab + 13 world names) ≈ 495 mp3，~15 MB。

| Slug（en + zh 共用） | en voice | zh-TW voice | 風格 |
|---|---|---|---|
| `young-aria` | en-US-AriaNeural | zh-TW-HsiaoChenNeural | 年輕女、姐姐 |
| `warm-jenny` | en-US-JennyNeural | zh-TW-HsiaoYuNeural | 溫暖女、媽媽 |
| `calm-guy` | en-US-GuyNeural | zh-TW-YunJheNeural | 低沉男、爸爸 |

## 跑法

- **Colab**：上傳 `words.json`（cell 3 會自動 prompt），跑完最後一格下載 `tts-zh-tw.zip`，解到 `~/Project/pig-recognition-assets/audio/tts-zh-tw/` 後 commit + tag v1.3.0。
- **本機 Jupyter**：自動讀 `~/Project/pig-recognition-assets/data/vocab-en-preschool/words.json`，直接寫到 repo 的 `audio/tts-zh-tw/`。

In [ ]:
!pip install -q edge-tts

## Step 1 · voice / 詞庫 / 世界 設定

STYLES 跟英文版 slug 一致，settings 切 voice 時 en + zh 同步切。
13 個世界 zh 名稱 hardcode（清單變動少）；152 個 vocab 從 JSON 讀。

In [ ]:
import json, os, sys
from pathlib import Path

# 3 voice preset — slug 對應英文版
STYLES = [
    {'slug': 'young-aria', 'voice': 'zh-TW-HsiaoChenNeural', 'rate': '+0%',  'pitch': '+2Hz'},
    {'slug': 'warm-jenny', 'voice': 'zh-TW-HsiaoYuNeural',   'rate': '-5%',  'pitch': '+0Hz'},
    {'slug': 'calm-guy',   'voice': 'zh-TW-YunJheNeural',    'rate': '-3%',  'pitch': '-1Hz'},
]

# 13 世界 zh 名稱（與 pig-english src/lib/game/worlds.ts 同步）
WORLDS = [
    ('living',       '客廳'),
    ('bedroom',      '臥室'),
    ('kitchen',      '廚房'),
    ('body',         '身體'),
    ('playground',   '遊樂場'),
    ('zoo',          '動物園'),
    ('transport',    '交通工具'),
    ('park',         '公園'),
    ('kindergarten', '幼兒園'),
    ('math',         '數學樂園'),
    ('farm',         '農場'),
    ('school',       '學校'),
    ('treehouse',    '彩虹樹屋'),
]

# 環境偵測：Colab → 上傳，本機 → 從 repo 路徑讀
IN_COLAB = 'google.colab' in sys.modules
if not IN_COLAB:
    try:
        from google.colab import files  # type: ignore
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

if IN_COLAB:
    from google.colab import files  # type: ignore
    print('🟢 在 Colab — 請上傳 words.json（從 ~/Project/pig-recognition-assets/data/vocab-en-preschool/）')
    uploaded = files.upload()
    vocab_data = json.loads(next(iter(uploaded.values())))
    OUTPUT_DIR = Path('tts-zh-tw')
else:
    REPO_ROOT = Path(os.environ.get('HOME', '/Users/jerry.wu')) / 'Project' / 'pig-recognition-assets'
    VOCAB_JSON = REPO_ROOT / 'data' / 'vocab-en-preschool' / 'words.json'
    if not VOCAB_JSON.exists():
        sys.exit(f'❌ 找不到 {VOCAB_JSON}，請先在 pig-english 跑 `npx tsx scripts/export_to_shared_assets.ts`')
    vocab_data = json.loads(VOCAB_JSON.read_text(encoding='utf-8'))
    OUTPUT_DIR = REPO_ROOT / 'audio' / 'tts-zh-tw'

vocab = vocab_data['words']
print(f'📖 詞庫：{len(vocab)} 字（version {vocab_data.get("version", "?")}）')
print(f'🌍 世界：{len(WORLDS)} 個')
print(f'🎤 voice 數：{len(STYLES)}')
print(f'📁 輸出：{OUTPUT_DIR}')

total_tasks = len(STYLES) * (len(vocab) + len(WORLDS))
print(f'⏳ 預估 {total_tasks} 個 mp3，Colab 跑 ~3-5 分鐘')

## Step 2 · 批次生成（含 retry + timeout）

Microsoft 免費 TTS 對並發敏感，BATCH=4 + 每 batch sleep 0.8s。
個別 task 20 秒 timeout、失敗 retry 3 次。跑完整個 cell 約 3-5 分鐘。

In [ ]:
import asyncio
import edge_tts

PER_TASK_TIMEOUT_S = 20
MAX_RETRIES = 3
RETRY_DELAY_S = 1.0
BATCH = 4
BATCH_SLEEP_S = 0.8

async def _save_with_timeout(text, voice, rate, pitch, out_path):
    communicate = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
    await asyncio.wait_for(communicate.save(str(out_path)), timeout=PER_TASK_TIMEOUT_S)

async def generate(text, voice, rate, pitch, out_path):
    if out_path.exists() and out_path.stat().st_size > 0:
        return 'skipped'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            await _save_with_timeout(text, voice, rate, pitch, out_path)
            return 'generated'
        except Exception as e:
            last_err = e
            if attempt < MAX_RETRIES:
                await asyncio.sleep(RETRY_DELAY_S * attempt)
    return f'error[{out_path.name}]: {last_err}'

# 蒐集所有 task spec
specs = []
for style in STYLES:
    for w in vocab:
        specs.append((w['zh'], style, OUTPUT_DIR / style['slug'] / 'words' / f"{w['id']}.mp3"))
    for wid, name in WORLDS:
        specs.append((name, style, OUTPUT_DIR / style['slug'] / 'worlds' / f"{wid}.mp3"))

print(f'共 {len(specs)} 個 task，BATCH={BATCH}，sleep={BATCH_SLEEP_S}s\n')

results = []
for i in range(0, len(specs), BATCH):
    batch = specs[i:i + BATCH]
    coros = [
        generate(text, style['voice'], style['rate'], style['pitch'], path)
        for text, style, path in batch
    ]
    batch_results = await asyncio.gather(*coros)
    results.extend(batch_results)
    done = i + len(batch)
    if done % 40 == 0 or done == len(specs):
        print(f'  進度 {done}/{len(specs)}')
    await asyncio.sleep(BATCH_SLEEP_S)

counts = {'generated': 0, 'skipped': 0, 'error': 0}
errors = []
for r in results:
    if r == 'generated': counts['generated'] += 1
    elif r == 'skipped': counts['skipped'] += 1
    else:
        counts['error'] += 1
        errors.append(r)

print(f"\n✔ generated: {counts['generated']}")
print(f"  skipped:   {counts['skipped']}")
print(f"  errors:    {counts['error']}")
if errors:
    print('\n⚠ 失敗清單（重跑這個 cell 會只補失敗的）：')
    for e in errors[:10]:
        print(' ', e)

## Step 3 · 試聽（任選 cell 跑）

In [ ]:
# Colab: 內嵌 audio player
if IN_COLAB:
    from IPython.display import Audio, display
    for style in STYLES:
        sample = OUTPUT_DIR / style['slug'] / 'words' / 'cat.mp3'
        if sample.exists():
            print(f"\n🎤 {style['slug']} ({style['voice']}) → cat.mp3")
            display(Audio(str(sample)))
else:
    import subprocess
    for style in STYLES:
        sample = OUTPUT_DIR / style['slug'] / 'words' / 'cat.mp3'
        if sample.exists():
            print(f"🎤 {style['slug']}: afplay {sample}")
    print('\n本機跑：自己挑一個 afplay 試聽。')

## Step 4 · 打包下載（Colab only）

跑完這格會下載 `tts-zh-tw.zip`，解壓到 `~/Project/pig-recognition-assets/audio/` 後 commit。
本機跑不用，mp3 已直接寫到 repo。

In [ ]:
if IN_COLAB:
    import shutil
    zip_path = '/content/tts-zh-tw'
    shutil.make_archive(zip_path, 'zip', str(OUTPUT_DIR))
    print(f'📦 打包完成：{zip_path}.zip')
    from google.colab import files  # type: ignore
    files.download(f'{zip_path}.zip')
else:
    print('本機跑：mp3 已直接在 repo，不用打包。')
    print(f'  cd ~/Project/pig-recognition-assets')
    print(f'  git add audio/tts-zh-tw scripts/gen-tts-zh')
    print(f'  git commit -m "Add zh-TW TTS: 3 voice × (152 vocab + 13 worlds)"')
    print(f'  git tag v1.3.0')
    print(f'  git push --tags')